# Funding Rate Mechanics

Demonstrates how KL divergence drives the funding rate and how it penalises positions that push the market away from the anchor.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))

import math
import numpy as np
import matplotlib.pyplot as plt

from pdamm.amm.gaussian_mixture import GaussianMixtureAMM, GaussianMixtureParams
from pdamm.amm.anchor import CompositeAnchor, OracleAnchor, kl_divergence
from pdamm.amm.oracle import ConstantOracle
from pdamm.amm.funding import FundingConfig
from pdamm.amm.perp_market import PerpMarket
from pdamm.agents.lp_provider import LPProvider
from pdamm.agents.noise_trader import NoiseTrader
from pdamm.agents.informed_trader import InformedTrader

K_RUST = 5.0 * math.sqrt(10.0 * math.sqrt(math.pi))

def make_perp(amm_mu=95.0, anchor_mu=95.0, funding_rate_bps=200.0, funding_interval=10):
    amm = GaussianMixtureAMM(
        b=50.0, k=K_RUST,
        params=GaussianMixtureParams.single(mu=amm_mu, sigma=10.0),
        max_collateral_per_trade=50.0,
    )
    oracle = ConstantOracle(mu=anchor_mu, sigma=5.0)
    anchor = CompositeAnchor(sub_anchors=[OracleAnchor(oracle_feed=oracle)])
    config = FundingConfig(
        funding_rate_bps=funding_rate_bps,
        slots_per_period=funding_interval,
        kl_cap=10.0, min_kl_threshold=0.001,
    )
    return PerpMarket(amm=amm, anchor=anchor, funding_config=config,
                      funding_interval=funding_interval)


## KL divergence vs. funding rate

In [ ]:
anchor_mu = 100.0
amm_mus = np.linspace(80, 130, 60)
kl_vals, rate_vals = [], []

for amm_mu in amm_mus:
    market = make_perp(amm_mu=amm_mu, anchor_mu=anchor_mu)
    kl_vals.append(market.current_kl_from_anchor())
    rate_vals.append(market.spot_funding_rate())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(amm_mus, kl_vals)
ax1.set_xlabel("AMM mu")
ax1.set_ylabel("KL(AMM || anchor)")
ax1.set_title("KL divergence from anchor")
ax1.axvline(anchor_mu, color="red", linestyle="--", label="Anchor mu")
ax1.legend()

ax2.plot(amm_mus, rate_vals)
ax2.set_xlabel("AMM mu")
ax2.set_ylabel("Funding rate (per slot)")
ax2.set_title("Spot funding rate vs AMM mu")
ax2.axvline(anchor_mu, color="red", linestyle="--", label="Anchor mu")
ax2.legend()
plt.tight_layout()
plt.show()


## Funding accumulation over time: noise vs. informed

In [ ]:
market = make_perp(anchor_mu=100.0, funding_rate_bps=300.0, funding_interval=5)
lp = LPProvider("lp0", initial_deposit=200.0)
lp.enter(market)

noise = NoiseTrader("noise", mu_shock_scale=8.0, seed=3)
informed = InformedTrader(
    "informed", GaussianMixtureParams.single(mu=100.0, sigma=9.0),
    step_fraction=0.15, max_fraction_of_b=0.10,
)

slot_hist, kl_hist, rate_hist, cash_hist = [], [], [], []
for step in range(200):
    noise.step(market)
    informed.step(market)
    market.tick(1)
    slot_hist.append(market.slot)
    kl_hist.append(market.current_kl_from_anchor())
    rate_hist.append(market.spot_funding_rate())
    cash_hist.append(market.amm.cash)

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
axes[0].plot(slot_hist, kl_hist, color="purple")
axes[0].set_ylabel("KL(AMM||anchor)")
axes[0].set_title("Funding dynamics: noise + informed traders")

axes[1].plot(slot_hist, rate_hist, color="orange")
axes[1].set_ylabel("Funding rate")

axes[2].plot(slot_hist, cash_hist, color="blue")
axes[2].set_ylabel("Vault cash")
axes[2].set_xlabel("Slot")
plt.tight_layout()
plt.show()

print(f"Final vault cash: {market.amm.cash:.3f}")
print(f"Total funding events: {len(market.funding_history)}")
